# 03 - Preprocessing

Prepare clean track-level data for content-based recommendation.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
MODEL_DIR = PROJECT_DIR / "models" / "content_based"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = PROCESSED_DIR / "dataset_cleaned.csv"

## Load Cleaned Data

In [2]:
df = pd.read_csv(DATA_PATH)
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
df.head()

Loaded: 113,999 rows x 20 columns


,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


## Keep One Row Per Track

In [3]:
tracks = df.drop_duplicates(subset="track_id").reset_index(drop=True).copy()
tracks["content_index"] = np.arange(len(tracks))

text_columns = ["artists", "album_name", "track_name", "track_genre"]
for column in text_columns:
    tracks[column] = tracks[column].fillna("unknown").astype(str).str.strip()

tracks["explicit"] = tracks["explicit"].astype(bool)

print(f"Unique tracks: {tracks.shape[0]:,}")
print(f"Duplicates removed: {len(df) - len(tracks):,}")

Unique tracks: 89,740
Duplicates removed: 24,259


## Define Content Features

In [4]:
metadata_columns = ["content_index", "track_id", "track_name", "artists", "album_name", "track_genre"]

numeric_features = [
    "popularity", "duration_ms", "danceability", "energy", "key", "loudness", "mode",
    "speechiness", "acousticness", "instrumentalness", "liveness", "valence", "tempo", "time_signature",
]

categorical_features = ["track_genre", "explicit"]
required = set(metadata_columns + numeric_features + categorical_features) - {"content_index"}
missing = sorted(required - set(tracks.columns))
assert not missing, f"Missing columns: {missing}"

track_metadata = tracks[metadata_columns].copy()
print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")

Numeric features: 14
Categorical features: 2


## Save Preprocessed Files

In [5]:
tracks_path = PROCESSED_DIR / "content_tracks_preprocessed.csv"
metadata_path = PROCESSED_DIR / "content_track_metadata.csv"
feature_config_path = MODEL_DIR / "content_feature_config.json"

tracks.to_csv(tracks_path, index=False)
track_metadata.to_csv(metadata_path, index=False)

feature_config = {
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "metadata_columns": metadata_columns,
}

with open(feature_config_path, "w", encoding="utf-8") as file:
    json.dump(feature_config, file, indent=2)

print("Saved preprocessing outputs.")
print(tracks_path)
print(metadata_path)
print(feature_config_path)

Saved preprocessing outputs.
d:\ml\sportify-recommendation\data\processed\content_tracks_preprocessed.csv
d:\ml\sportify-recommendation\data\processed\content_track_metadata.csv
d:\ml\sportify-recommendation\models\content_based\content_feature_config.json


## Summary

In [6]:
summary = pd.DataFrame({
    "item": ["cleaned_rows", "unique_tracks", "numeric_features", "categorical_features"],
    "value": [len(df), len(tracks), len(numeric_features), len(categorical_features)],
})
summary

,item,value
0,cleaned_rows,113999
1,unique_tracks,89740
2,numeric_features,14
3,categorical_features,2
